# MSD Liver Tumor Classification — Cloud GPU Training

**Project:** Multi-Model Unified Organ Cancer Identifier  
**Author:** Sanchit Singh (2401020387@cgu-odisha.ac.in)  
**Platform:** AI Kosh (AIRAWAT)  

This notebook trains two 3D CNN models (ResNet3D and VGG3D) on pre-extracted
96x96x96 CT patches from the MSD Task03 Liver dataset.

**Prerequisites:** Upload the following before running:
- `src/` folder (all .py files)
- `data/liver_patches/` folder (~17.8 GB of .npy patches + manifest.csv)

**Expected runtime:** 6-12 hours

---
## 1. Environment Check

In [ ]:
import torch
import sys
import os

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)
print(f"Python:        {sys.version.split()[0]}")
print(f"PyTorch:       {torch.__version__}")
print(f"CUDA avail:    {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU:           {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"VRAM:          {vram:.1f} GB")
else:
    print("WARNING: No GPU detected! Training will be extremely slow.")

print(f"Working dir:   {os.getcwd()}")
print("=" * 60)

## 2. Install Dependencies (if needed)

AI Kosh likely has PyTorch pre-installed. Run this cell to install any missing packages.

In [ ]:
!pip install scikit-learn matplotlib numpy scipy SimpleITK nibabel -q

## 3. Verify Uploaded Files

Adjust `PROJECT_ROOT` below if your files are in a different location.

In [ ]:
# ── Adjust this path if your project is in a different directory ──
PROJECT_ROOT = os.getcwd()
# ─────────────────────────────────────────────────────────────────

src_dir = os.path.join(PROJECT_ROOT, "src")
patches_dir = os.path.join(PROJECT_ROOT, "data", "liver_patches")
manifest_path = os.path.join(patches_dir, "manifest.csv")

print("Checking uploaded files...")
print()

# Check src/
required_files = [
    "main_liver.py", "architecture.py", "training.py",
    "evaluator.py", "fast_dataset.py", "gradcam.py", "utils.py"
]
missing = [f for f in required_files if not os.path.isfile(os.path.join(src_dir, f))]
if missing:
    print(f"MISSING source files: {missing}")
    print(f"Expected in: {src_dir}")
else:
    print(f"  src/ .............. OK ({len(required_files)} files found)")

# Check patches
if os.path.isfile(manifest_path):
    import csv
    with open(manifest_path) as f:
        n_patches = sum(1 for _ in csv.DictReader(f))
    print(f"  manifest.csv ...... OK ({n_patches} patches)")
else:
    print(f"MISSING: {manifest_path}")
    print("Upload data/liver_patches/ with manifest.csv and .npy files")

# Check a few .npy files exist
npy_files = [f for f in os.listdir(patches_dir) if f.endswith(".npy")] if os.path.isdir(patches_dir) else []
if npy_files:
    print(f"  .npy patches ...... OK ({len(npy_files)} files)")
    # Check patch dimensions
    import numpy as np
    sample = np.load(os.path.join(patches_dir, npy_files[0]))
    print(f"  Patch shape ....... {sample.shape} (dtype: {sample.dtype})")
else:
    print("MISSING: No .npy patch files found!")

print()
print("If everything shows OK, proceed to the next cell.")

## 4. Run Training

This cell runs the full training pipeline:
- ResNet3D (3.62M params) — 3-fold cross-validation
- VGG3D (0.87M params) — 3-fold cross-validation
- Grad-CAM visualizations after fold 0
- Cross-model comparison plots

**Estimated time: 6-12 hours.** Progress is printed continuously.

The cell uses `%run` so that if the kernel disconnects, you can check
`results/liver/` for any saved checkpoints and plots.

In [ ]:
# Add src/ to Python path so imports work
import sys
src_path = os.path.join(PROJECT_ROOT, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Run the full training pipeline
%run {src_path}/main_liver.py

## 5. Verify Results

After training completes, check that all output files were generated.

In [ ]:
results_dir = os.path.join(PROJECT_ROOT, "results", "liver")
print("Generated files:")
print("=" * 60)

for root, dirs, files in os.walk(results_dir):
    level = root.replace(results_dir, "").count(os.sep)
    indent = "  " * level
    folder = os.path.basename(root)
    print(f"{indent}{folder}/")
    for f in sorted(files):
        fpath = os.path.join(root, f)
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"{indent}  {f} ({size_mb:.1f} MB)")

## 6. Display Results Summary

In [ ]:
summary_path = os.path.join(results_dir, "plots", "results_summary.txt")
if os.path.isfile(summary_path):
    with open(summary_path) as f:
        print(f.read())
else:
    print("results_summary.txt not found -- training may not have completed.")

## 7. Display Plots

In [ ]:
from IPython.display import display, Image as IPImage
import glob

plot_dir = os.path.join(results_dir, "plots")

# ROC comparison
roc_path = os.path.join(plot_dir, "comparison_roc.png")
if os.path.isfile(roc_path):
    print("Comparison ROC Curve:")
    display(IPImage(filename=roc_path, width=700))

# Per-model plots
for model in ["resnet3d", "vgg3d"]:
    for plot_type in ["roc_curve_kfold.png", "training_curves_kfold.png"]:
        p = os.path.join(plot_dir, model, plot_type)
        if os.path.isfile(p):
            print(f"\n{model} - {plot_type}:")
            display(IPImage(filename=p, width=700))

# Grad-CAM samples
print("\nGrad-CAM Visualizations:")
for model in ["vgg3d", "resnet3d"]:
    gcam_dir = os.path.join(plot_dir, "gradcam", model)
    if os.path.isdir(gcam_dir):
        print(f"\n--- {model} ---")
        for img in sorted(glob.glob(os.path.join(gcam_dir, "*.png")))[:4]:
            display(IPImage(filename=img, width=500))

## 8. Download Results

Run this cell to zip all results for easy download.

In [ ]:
import shutil

zip_name = "liver_results"
zip_path = shutil.make_archive(
    os.path.join(PROJECT_ROOT, zip_name),
    'zip',
    root_dir=os.path.join(PROJECT_ROOT, "results"),
    base_dir="liver"
)

size_mb = os.path.getsize(zip_path) / 1e6
print(f"Results archived: {zip_path} ({size_mb:.1f} MB)")
print("Download this file from the file browser on the left.")